# Amazon ML Challenge: Entity Resolution Pipeline

This notebook orchestrates the complete End-to-End Multilingual Entity Resolution pipeline inside **AWS SageMaker**.

### Pipeline Overview:
1. **Preprocessing & Normalization**: Safe Unicode NFKC normalization and script-aware Indic transliteration (`ai4bharat-transliteration` / `indic-transliteration`).
2. **Candidate Blocking**: Country-compatible candidate pair generation with string similarity filtering.
3. **Feature Engineering**: Multi-field pairwise similarity vector extraction (Levenshtein, Jaro-Winkler, token sort/set ratio, character n-grams, digit overlap).
4. **Model Training & Optimization**: Gradient Boosted Decision Tree (`HistGradientBoostingClassifier`) with threshold tuning for Macro $F_{0.5}$.
5. **Inference & Submission Formatting**: Full test set entity resolution export matching strict format specifications.

In [ ]:
# Install required packages in SageMaker environment if needed
!pip install -q pandas numpy scikit-learn python-Levenshtein pyyaml indic-transliteration ai4bharat-transliteration

import sys
import os
from pathlib import Path

# Download and extract dataset from S3 bucket if not present locally
S3_BUCKET_NAME = "amazon-ml-211125709069-ap-south-1-an"
dataset_path = Path("student_resource/dataset")
if not (dataset_path / "train").exists() or not (dataset_path / "test").exists():
    print(f"Dataset files not found locally. Downloading dataset.zip from s3://{S3_BUCKET_NAME}...")
    !aws s3 cp s3://{S3_BUCKET_NAME}/dataset.zip dataset.zip
    !unzip -q -o dataset.zip -d student_resource/
    print("Dataset successfully downloaded and extracted.")
else:
    print("Dataset directory exists locally.")

# Ensure workspace root is in python path
workspace_dir = os.path.abspath(".")
if workspace_dir not in sys.path:
    sys.path.append(workspace_dir)

print("Python Path configured.")

In [ ]:
# Test Script-Aware Indic Transliteration
from src.preprocessing.transliteration import ScriptAwareTransliterator

transliterator = ScriptAwareTransliterator()
sample_texts = ["नमस्ते दुनिया", "தமிழ் நாடு", "Bengaluru"]

for text in sample_texts:
    res = transliterator.transliterate_text(text)
    print(f"Original: '{text}' -> Transliterated: '{res}'")

In [ ]:
# Run Synthetic Pipeline Test (Lightweight Demo)
from src.data.schema import BusinessRecord
from src.preprocessing.normalize import UnicodeNormalizer
from src.blocking.candidate_generator import SimpleCandidateGenerator
from src.features.pair_features import PairFeatureExtractor
from src.models.matcher import EntityMatcher

s1 = [BusinessRecord("s1_1", "Amazon India Pvt Ltd", "Bangalore, KA", "IN", "s1")]
s2 = [BusinessRecord("s2_1", "Amazon India Private Limited", "Bengaluru, Karnataka", "IN", "s2")]
s3 = [BusinessRecord("s3_1", "Amazon Retail", "Mumbai", "IN", "s3")]

normalizer = UnicodeNormalizer()
s1_n = [transliterator.transliterate_record(r) for r in normalizer.normalize_dataset(s1)]
s2_n = [transliterator.transliterate_record(r) for r in normalizer.normalize_dataset(s2)]
s3_n = [transliterator.transliterate_record(r) for r in normalizer.normalize_dataset(s3)]

gen = SimpleCandidateGenerator()
pairs = gen.generate_candidates(s1_n, s2_n, s3_n)
print(f"Synthetic candidate pairs generated: {len(pairs)}")

extractor = PairFeatureExtractor()
feats = extractor.extract_batch_features(pairs)
print("Feature extraction completed successfully:")
print(feats)

## Full Training Pipeline Execution (AWS SageMaker)
Execute full model training on the training dataset.

In [ ]:
from src.train import train

# Execute full training on train dataset
training_metrics = train(config="configs/config.yaml")
print("Training completed with metrics:", training_metrics)

## Full Test Dataset Inference Execution
Execute candidate generation, ML scoring, cluster formation, and export TSV submission files.

In [ ]:
from src.inference import run_inference

# Execute inference on test dataset
run_inference(config="configs/config.yaml")
print("Inference execution completed successfully.")

## Submission Validation
Verify generated submission files against challenge validation script rules.

In [ ]:
!python student_resource/utils/validate_submission.py --matching_path output/matching_results.tsv --candidate_path output/candidate_pairs.tsv